In [1]:
import numpy as np
import pandas as pd
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

import seaborn as sns
import matplotlib.pyplot as plt
import os
from datetime import date
import datetime

# Generating the training data for the Heat and Diffusion Model

In [2]:
data_dir = "./1D-AEMpy/"
depth_steps = 21 * 2 

print(os.getcwd())

D:\projects\1D-AEMpy\mcl\1_trainingData-MLP


In [3]:
meterological_data_df = pd.read_csv("./../output/lakes/erken/output/py_meteorology_input.csv")
meterological_data_df = meterological_data_df # considering everything from 2nd time step

num_time_steps = meterological_data_df.shape[0]
depth_list = np.array(list(range(0, depth_steps)) * num_time_steps)*0.5+.25
depth_df = pd.DataFrame(data={'depth':depth_list})

#repeating the dataframe depth_steps number of times
meterological_data_df = pd.DataFrame(np.repeat(meterological_data_df.values, depth_steps, axis=0), columns=meterological_data_df.columns)
meterological_data_df = pd.concat([depth_df, meterological_data_df], ignore_index=False, axis=1)
meterological_data_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,icemovAvg,density_snow,ice_prior,snow_prior,snowice_prior,rho_snow_prior,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior
0,0.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,6.546754,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.546754
1,0.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,6.546754,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.546754
2,1.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,6.546754,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.546754
3,1.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,6.546754,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.546754
4,2.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,6.546754,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.546754
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1839595,18.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,23.140659,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,23.140659
1839596,19.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,23.140659,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,23.140659
1839597,19.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,23.140659,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,23.140659
1839598,20.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,23.140659,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,23.140659


In [4]:
# Input INITIAL TEMP 00

out_temp_df = pd.read_csv("./../output/lakes/erken/output/py_temp_initial00.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'temp_init00':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

,time,temp_init00,depth
0,2017-06-28 20:00:00,16.810400,0.25
1,2017-06-28 20:00:00,16.810400,0.75
2,2017-06-28 20:00:00,16.810400,1.25
3,2017-06-28 20:00:00,16.814190,1.75
4,2017-06-28 20:00:00,16.825920,2.25
...,...,...,...
1839595,2022-06-27 19:00:00,13.245690,18.75
1839596,2022-06-27 19:00:00,13.252646,19.25
1839597,2022-06-27 19:00:00,13.258238,19.75
1839598,2022-06-27 19:00:00,13.265651,20.25


In [5]:
final_df = meterological_data_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,density_snow,ice_prior,snow_prior,snowice_prior,rho_snow_prior,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00
0,0.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.546754,16.810400
1,0.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.546754,16.810400
2,1.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.546754,16.810400
3,1.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.546754,16.814190
4,2.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.546754,16.825920
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1839595,18.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,23.140659,13.245690
1839596,19.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,23.140659,13.252646
1839597,19.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,23.140659,13.258238
1839598,20.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,23.140659,13.265651


In [6]:
# Input HEAT TEMP 01

out_temp_df = pd.read_csv("./../output/lakes/erken/output/py_temp_heat01.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'temp_heat01':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,ice_prior,snow_prior,snowice_prior,rho_snow_prior,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01
0,0.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.546754,16.810400,16.497682
1,0.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.546754,16.810400,16.834485
2,1.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.546754,16.810400,16.828012
3,1.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.546754,16.814190,16.827070
4,2.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.546754,16.825920,16.835371
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1839595,18.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,23.140659,13.245690,13.245754
1839596,19.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,23.140659,13.252646,13.252744
1839597,19.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,23.140659,13.258238,13.258464
1839598,20.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,23.140659,13.265651,13.265767


In [7]:
# Input ICE TEMP 02

out_temp_df = pd.read_csv("./../output/lakes/erken/output/py_temp_ice02.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'temp_ice02':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,snow_prior,snowice_prior,rho_snow_prior,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02
0,0.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,0.0,250.0,1.0,0.0,0.8,6.546754,16.810400,16.497682,16.497682
1,0.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,0.0,250.0,1.0,0.0,0.8,6.546754,16.810400,16.834485,16.834485
2,1.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,0.0,250.0,1.0,0.0,0.8,6.546754,16.810400,16.828012,16.828012
3,1.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,0.0,250.0,1.0,0.0,0.8,6.546754,16.814190,16.827070,16.827070
4,2.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,0.0,250.0,1.0,0.0,0.8,6.546754,16.825920,16.835371,16.835371
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1839595,18.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.0,0.0,250.0,1.0,0.0,0.8,23.140659,13.245690,13.245754,13.245754
1839596,19.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.0,0.0,250.0,1.0,0.0,0.8,23.140659,13.252646,13.252744,13.252744
1839597,19.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.0,0.0,250.0,1.0,0.0,0.8,23.140659,13.258238,13.258464,13.258464
1839598,20.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.0,0.0,250.0,1.0,0.0,0.8,23.140659,13.265651,13.265767,13.265767


In [8]:
# Input DIFF TEMP 03

out_temp_df = pd.read_csv("./../output/lakes/erken/output/py_temp_diff03.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'temp_diff03':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,snowice_prior,rho_snow_prior,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02,temp_diff03
0,0.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,250.0,1.0,0.0,0.8,6.546754,16.810400,16.497682,16.497682,16.497682
1,0.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,250.0,1.0,0.0,0.8,6.546754,16.810400,16.834485,16.834485,16.833785
2,1.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,250.0,1.0,0.0,0.8,6.546754,16.810400,16.828012,16.828012,16.828023
3,1.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,250.0,1.0,0.0,0.8,6.546754,16.814190,16.827070,16.827070,16.827089
4,2.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,250.0,1.0,0.0,0.8,6.546754,16.825920,16.835371,16.835371,16.835365
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1839595,18.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.0,250.0,1.0,0.0,0.8,23.140659,13.245690,13.245754,13.245754,13.245743
1839596,19.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.0,250.0,1.0,0.0,0.8,23.140659,13.252646,13.252744,13.252744,13.252728
1839597,19.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.0,250.0,1.0,0.0,0.8,23.140659,13.258238,13.258464,13.258464,13.258453
1839598,20.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.0,250.0,1.0,0.0,0.8,23.140659,13.265651,13.265767,13.265767,13.267746


In [9]:
# Input CONV TEMP 04

out_temp_df = pd.read_csv("./../output/lakes/erken/output/py_temp_conv04.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'temp_conv04':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,rho_snow_prior,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02,temp_diff03,temp_conv04
0,0.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,250.0,1.0,0.0,0.8,6.546754,16.810400,16.497682,16.497682,16.497682,16.775408
1,0.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,250.0,1.0,0.0,0.8,6.546754,16.810400,16.834485,16.834485,16.833785,16.780231
2,1.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,250.0,1.0,0.0,0.8,6.546754,16.810400,16.828012,16.828012,16.828023,16.785423
3,1.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,250.0,1.0,0.0,0.8,6.546754,16.814190,16.827070,16.827070,16.827089,16.790182
4,2.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,250.0,1.0,0.0,0.8,6.546754,16.825920,16.835371,16.835371,16.835365,16.794665
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1839595,18.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,250.0,1.0,0.0,0.8,23.140659,13.245690,13.245754,13.245754,13.245743,13.245743
1839596,19.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,250.0,1.0,0.0,0.8,23.140659,13.252646,13.252744,13.252744,13.252728,13.252728
1839597,19.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,250.0,1.0,0.0,0.8,23.140659,13.258238,13.258464,13.258464,13.258453,13.260375
1839598,20.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,250.0,1.0,0.0,0.8,23.140659,13.265651,13.265767,13.265767,13.267746,13.260375


In [10]:
# Input MIX TEMP 05

out_temp_df = pd.read_csv("./../output/lakes/erken/output/py_temp_mix05.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'temp_mix05':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02,temp_diff03,temp_conv04,temp_mix05
0,0.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,1.0,0.0,0.8,6.546754,16.810400,16.497682,16.497682,16.497682,16.775408,16.787306
1,0.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,1.0,0.0,0.8,6.546754,16.810400,16.834485,16.834485,16.833785,16.780231,16.787306
2,1.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,1.0,0.0,0.8,6.546754,16.810400,16.828012,16.828012,16.828023,16.785423,16.787306
3,1.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,1.0,0.0,0.8,6.546754,16.814190,16.827070,16.827070,16.827089,16.790182,16.787306
4,2.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,1.0,0.0,0.8,6.546754,16.825920,16.835371,16.835371,16.835365,16.794665,16.787306
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1839595,18.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,1.0,0.0,0.8,23.140659,13.245690,13.245754,13.245754,13.245743,13.245743,13.245743
1839596,19.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,1.0,0.0,0.8,23.140659,13.252646,13.252744,13.252744,13.252728,13.252728,13.252728
1839597,19.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,1.0,0.0,0.8,23.140659,13.258238,13.258464,13.258464,13.258453,13.260375,13.260375
1839598,20.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,1.0,0.0,0.8,23.140659,13.265651,13.265767,13.265767,13.267746,13.260375,13.260375


In [11]:
# Input BUOYANCY

out_temp_df = pd.read_csv("./../output/lakes/erken/output/py_buoyancy.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'buoyancy':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02,temp_diff03,temp_conv04,temp_mix05,buoyancy
0,0.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,0.8,6.546754,16.810400,16.497682,16.497682,16.497682,16.775408,16.787306,0.000000
1,0.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,0.8,6.546754,16.810400,16.834485,16.834485,16.833785,16.780231,16.787306,0.000000
2,1.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,0.8,6.546754,16.810400,16.828012,16.828012,16.828023,16.785423,16.787306,0.000000
3,1.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,0.8,6.546754,16.814190,16.827070,16.827070,16.827089,16.790182,16.787306,0.000000
4,2.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.0,0.8,6.546754,16.825920,16.835371,16.835371,16.835365,16.794665,16.787306,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1839595,18.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.0,0.8,23.140659,13.245690,13.245754,13.245754,13.245743,13.245743,13.245743,0.000018
1839596,19.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.0,0.8,23.140659,13.252646,13.252744,13.252744,13.252728,13.252728,13.252728,0.000020
1839597,19.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.0,0.8,23.140659,13.258238,13.258464,13.258464,13.258453,13.260375,13.260375,0.000000
1839598,20.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.0,0.8,23.140659,13.265651,13.265767,13.265767,13.267746,13.260375,13.260375,0.012154


In [12]:
# Input DIFFUSIVITY

out_temp_df = pd.read_csv("./../output/lakes/erken/output/py_diff.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'diffusivity':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02,temp_diff03,temp_conv04,temp_mix05,buoyancy,diffusivity
0,0.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.8,6.546754,16.810400,16.497682,16.497682,16.497682,16.775408,16.787306,0.000000,1.410307e-07
1,0.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.8,6.546754,16.810400,16.834485,16.834485,16.833785,16.780231,16.787306,0.000000,1.401284e-07
2,1.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.8,6.546754,16.810400,16.828012,16.828012,16.828023,16.785423,16.787306,0.000000,1.400089e-07
3,1.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.8,6.546754,16.814190,16.827070,16.827070,16.827089,16.790182,16.787306,0.000000,1.400005e-07
4,2.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.8,6.546754,16.825920,16.835371,16.835371,16.835365,16.794665,16.787306,0.000000,1.400000e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1839595,18.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.8,23.140659,13.245690,13.245754,13.245754,13.245743,13.245743,13.245743,0.000018,2.800006e-07
1839596,19.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.8,23.140659,13.252646,13.252744,13.252744,13.252728,13.252728,13.252728,0.000020,2.800004e-07
1839597,19.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.8,23.140659,13.258238,13.258464,13.258464,13.258453,13.260375,13.260375,0.000000,2.800003e-07
1839598,20.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.8,23.140659,13.265651,13.265767,13.265767,13.267746,13.260375,13.260375,0.012154,2.800000e-07


In [13]:
# Input density gradient

out_temp_df = pd.read_csv("./../output/lakes/erken/output/py_density-conv.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'densityGradient':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02,temp_diff03,temp_conv04,temp_mix05,buoyancy,diffusivity,densityGradient
0,0.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,6.546754,16.810400,16.497682,16.497682,16.497682,16.775408,16.787306,0.000000,1.410307e-07,0.057128
1,0.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,6.546754,16.810400,16.834485,16.834485,16.833785,16.780231,16.787306,0.000000,1.401284e-07,-0.000990
2,1.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,6.546754,16.810400,16.828012,16.828012,16.828023,16.785423,16.787306,0.000000,1.400089e-07,-0.000160
3,1.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,6.546754,16.814190,16.827070,16.827070,16.827089,16.790182,16.787306,0.000000,1.400005e-07,0.001422
4,2.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,6.546754,16.825920,16.835371,16.835371,16.835365,16.794665,16.787306,0.000000,1.400000e-07,0.000891
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1839595,18.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,23.140659,13.245690,13.245754,13.245754,13.245743,13.245743,13.245743,0.000018,2.800006e-07,0.000906
1839596,19.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,23.140659,13.252646,13.252744,13.252744,13.252728,13.252728,13.252728,0.000020,2.800004e-07,0.000743
1839597,19.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,23.140659,13.258238,13.258464,13.258464,13.258453,13.260375,13.260375,0.000000,2.800003e-07,0.001207
1839598,20.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,23.140659,13.265651,13.265767,13.265767,13.267746,13.260375,13.260375,0.012154,2.800000e-07,-0.619561


In [14]:
# Input temp change

out_temp_df = pd.read_csv("./../output/lakes/erken/output/py_temp-conv.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'tempChange':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,temp_init00,temp_heat01,temp_ice02,temp_diff03,temp_conv04,temp_mix05,buoyancy,diffusivity,densityGradient,tempChange
0,0.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,16.810400,16.497682,16.497682,16.497682,16.775408,16.787306,0.000000,1.410307e-07,0.057128,16.833785
1,0.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,16.810400,16.834485,16.834485,16.833785,16.780231,16.787306,0.000000,1.401284e-07,-0.000990,16.828023
2,1.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,16.810400,16.828012,16.828012,16.828023,16.785423,16.787306,0.000000,1.400089e-07,-0.000160,16.827089
3,1.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,16.814190,16.827070,16.827070,16.827089,16.790182,16.787306,0.000000,1.400005e-07,0.001422,16.835365
4,2.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,16.825920,16.835371,16.835371,16.835365,16.794665,16.787306,0.000000,1.400000e-07,0.000891,16.840548
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1839595,18.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,13.245690,13.245754,13.245754,13.245743,13.245743,13.245743,0.000018,2.800006e-07,0.000906,13.252728
1839596,19.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,13.252646,13.252744,13.252744,13.252728,13.252728,13.252728,0.000020,2.800004e-07,0.000743,13.258453
1839597,19.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,13.258238,13.258464,13.258464,13.258453,13.260375,13.260375,0.000000,2.800003e-07,0.001207,13.267746
1839598,20.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,13.265651,13.265767,13.265767,13.267746,13.260375,13.260375,0.012154,2.800000e-07,-0.619561,5.136564


In [15]:
# ICE AND SNOW

ice_data_df = pd.read_csv("./../output/lakes/erken/output/py_icesnow.csv")

#repeating the dataframe depth_steps number of times
ice_data_df = pd.DataFrame(np.repeat(ice_data_df.values, depth_steps, axis=0), columns=ice_data_df.columns)
ice_data_df = pd.concat([depth_df, ice_data_df], ignore_index=False, axis=1)
print(ice_data_df)

final_df = final_df.merge(ice_data_df, how='inner', on=['time','depth'])
final_df

         depth                 time  ice snow snowice
0         0.25  2017-06-28 20:00:00  0.0  0.0     0.0
1         0.75  2017-06-28 20:00:00  0.0  0.0     0.0
2         1.25  2017-06-28 20:00:00  0.0  0.0     0.0
3         1.75  2017-06-28 20:00:00  0.0  0.0     0.0
4         2.25  2017-06-28 20:00:00  0.0  0.0     0.0
...        ...                  ...  ...  ...     ...
1839595  18.75  2022-06-27 19:00:00  0.0  0.0     0.0
1839596  19.25  2022-06-27 19:00:00  0.0  0.0     0.0
1839597  19.75  2022-06-27 19:00:00  0.0  0.0     0.0
1839598  20.25  2022-06-27 19:00:00  0.0  0.0     0.0
1839599  20.75  2022-06-27 19:00:00  0.0  0.0     0.0

[1839600 rows x 5 columns]


,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,temp_diff03,temp_conv04,temp_mix05,buoyancy,diffusivity,densityGradient,tempChange,ice,snow,snowice
0,0.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,16.497682,16.775408,16.787306,0.000000,1.410307e-07,0.057128,16.833785,0.0,0.0,0.0
1,0.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,16.833785,16.780231,16.787306,0.000000,1.401284e-07,-0.000990,16.828023,0.0,0.0,0.0
2,1.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,16.828023,16.785423,16.787306,0.000000,1.400089e-07,-0.000160,16.827089,0.0,0.0,0.0
3,1.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,16.827089,16.790182,16.787306,0.000000,1.400005e-07,0.001422,16.835365,0.0,0.0,0.0
4,2.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,16.835365,16.794665,16.787306,0.000000,1.400000e-07,0.000891,16.840548,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1839595,18.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,13.245743,13.245743,13.245743,0.000018,2.800006e-07,0.000906,13.252728,0.0,0.0,0.0
1839596,19.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,13.252728,13.252728,13.252728,0.000020,2.800004e-07,0.000743,13.258453,0.0,0.0,0.0
1839597,19.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,13.258453,13.260375,13.260375,0.000000,2.800003e-07,0.001207,13.267746,0.0,0.0,0.0
1839598,20.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,13.267746,13.260375,13.260375,0.012154,2.800000e-07,-0.619561,5.136564,0.0,0.0,0.0


In [16]:
# lake characteristics

ice_data_df = pd.read_csv("./../output/lakes/erken/output/py_lakecharacteristics.csv")

#repeating the dataframe depth_steps number of times
ice_data_df = pd.DataFrame(np.repeat(ice_data_df.values, depth_steps, axis=0), columns=ice_data_df.columns)
ice_data_df = pd.concat([depth_df, ice_data_df], ignore_index=False, axis=1)
print(ice_data_df)

final_df = final_df.merge(ice_data_df, how='inner', on=['time','depth'])
final_df

         depth                 time     Volume_m2    Osgood MaxDepth_m  \
0         0.25  2017-06-28 20:00:00  213626250.75  2.855057      21.75   
1         0.75  2017-06-28 20:00:00  213626250.75  2.855057      21.75   
2         1.25  2017-06-28 20:00:00  213626250.75  2.855057      21.75   
3         1.75  2017-06-28 20:00:00  213626250.75  2.855057      21.75   
4         2.25  2017-06-28 20:00:00  213626250.75  2.855057      21.75   
...        ...                  ...           ...       ...        ...   
1839595  18.75  2022-06-27 19:00:00  213626250.75  2.855057      21.75   
1839596  19.25  2022-06-27 19:00:00  213626250.75  2.855057      21.75   
1839597  19.75  2022-06-27 19:00:00  213626250.75  2.855057      21.75   
1839598  20.25  2022-06-27 19:00:00  213626250.75  2.855057      21.75   
1839599  20.75  2022-06-27 19:00:00  213626250.75  2.855057      21.75   

        MeanDepth_m  
0          10.02519  
1          10.02519  
2          10.02519  
3          10.02519  
4

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,diffusivity,densityGradient,tempChange,ice,snow,snowice,Volume_m2,Osgood,MaxDepth_m,MeanDepth_m
0,0.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,1.410307e-07,0.057128,16.833785,0.0,0.0,0.0,213626250.75,2.855057,21.75,10.02519
1,0.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,1.401284e-07,-0.000990,16.828023,0.0,0.0,0.0,213626250.75,2.855057,21.75,10.02519
2,1.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,1.400089e-07,-0.000160,16.827089,0.0,0.0,0.0,213626250.75,2.855057,21.75,10.02519
3,1.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,1.400005e-07,0.001422,16.835365,0.0,0.0,0.0,213626250.75,2.855057,21.75,10.02519
4,2.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,1.400000e-07,0.000891,16.840548,0.0,0.0,0.0,213626250.75,2.855057,21.75,10.02519
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1839595,18.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,2.800006e-07,0.000906,13.252728,0.0,0.0,0.0,213626250.75,2.855057,21.75,10.02519
1839596,19.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,2.800004e-07,0.000743,13.258453,0.0,0.0,0.0,213626250.75,2.855057,21.75,10.02519
1839597,19.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,2.800003e-07,0.001207,13.267746,0.0,0.0,0.0,213626250.75,2.855057,21.75,10.02519
1839598,20.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,2.800000e-07,-0.619561,5.136564,0.0,0.0,0.0,213626250.75,2.855057,21.75,10.02519


In [17]:
temp_obs_df = pd.read_csv("./../output/lakes/erken/output/py_observed_temp.csv")


flattened_temp = temp_obs_df.iloc[:,1:].to_numpy().flatten()
time_stamp = temp_obs_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'obs_temp':flattened_temp, 'depth':depth_list}

temp_obs_df = pd.DataFrame(data=data)

temp_obs_df


final_df = final_df.merge(temp_obs_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,densityGradient,tempChange,ice,snow,snowice,Volume_m2,Osgood,MaxDepth_m,MeanDepth_m,obs_temp
0,0.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.057128,16.833785,0.0,0.0,0.0,213626250.75,2.855057,21.75,10.02519,16.81040
1,0.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,-0.000990,16.828023,0.0,0.0,0.0,213626250.75,2.855057,21.75,10.02519,16.81040
2,1.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,-0.000160,16.827089,0.0,0.0,0.0,213626250.75,2.855057,21.75,10.02519,16.81040
3,1.75,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.001422,16.835365,0.0,0.0,0.0,213626250.75,2.855057,21.75,10.02519,16.81419
4,2.25,2017-06-28 20:00:00,14.284031,-101.205668,-80.463293,-9.855218,65.608673,0.63,1296111.734899,0.00614,...,0.000891,16.840548,0.0,0.0,0.0,213626250.75,2.855057,21.75,10.02519,16.82592
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1839595,18.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.000906,13.252728,0.0,0.0,0.0,213626250.75,2.855057,21.75,10.02519,12.28217
1839596,19.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.000743,13.258453,0.0,0.0,0.0,213626250.75,2.855057,21.75,10.02519,12.28217
1839597,19.75,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,0.001207,13.267746,0.0,0.0,0.0,213626250.75,2.855057,21.75,10.02519,12.28217
1839598,20.25,2022-06-27 19:00:00,29.877803,7.785549,-9.789293,28.022921,135.648458,0.63,11444774.770452,0.026217,...,-0.619561,5.136564,0.0,0.0,0.0,213626250.75,2.855057,21.75,10.02519,12.28217


In [18]:
obs_array = final_df['obs_temp']
obs_array[obs_array == -999] = final_df['temp_mix05']
print(obs_array)
final_df['obs_temp'] = obs_array

0          16.81040
1          16.81040
2          16.81040
3          16.81419
4          16.82592
             ...   
1839595    12.28217
1839596    12.28217
1839597    12.28217
1839598    12.28217
1839599    12.28217
Name: obs_temp, Length: 1839600, dtype: float64


C:\Users\au740615\AppData\Local\Temp\ipykernel_18056\349148568.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  obs_array[obs_array == -999] = final_df['temp_mix05']


In [19]:
final_df_null = final_df.fillna('')
print(final_df_null.head)

<bound method NDFrame.head of          depth                 time  AirTemp_degC  Longwave_Wm-2  Latent_Wm-2  \
0         0.25  2017-06-28 20:00:00     14.284031    -101.205668   -80.463293   
1         0.75  2017-06-28 20:00:00     14.284031    -101.205668   -80.463293   
2         1.25  2017-06-28 20:00:00     14.284031    -101.205668   -80.463293   
3         1.75  2017-06-28 20:00:00     14.284031    -101.205668   -80.463293   
4         2.25  2017-06-28 20:00:00     14.284031    -101.205668   -80.463293   
...        ...                  ...           ...            ...          ...   
1839595  18.75  2022-06-27 19:00:00     29.877803       7.785549    -9.789293   
1839596  19.25  2022-06-27 19:00:00     29.877803       7.785549    -9.789293   
1839597  19.75  2022-06-27 19:00:00     29.877803       7.785549    -9.789293   
1839598  20.25  2022-06-27 19:00:00     29.877803       7.785549    -9.789293   
1839599  20.75  2022-06-27 19:00:00     29.877803       7.785549    -9.789293  

In [20]:
# iterating the columns
for col in final_df_null.columns:
    print(col)

depth
time
AirTemp_degC
Longwave_Wm-2
Latent_Wm-2
Sensible_Wm-2
Shortwave_Wm-2
lightExtinct_m-1
TKE_Jm-1
ShearStress_Nm-2
Area_m2
CC
ea
Jlw
Uw
Pa
RH
PP
IceSnowAttCoeff
iceFlag
icemovAvg
density_snow
ice_prior
snow_prior
snowice_prior
rho_snow_prior
IceSnowAttCoeff_prior
iceFlag_prior
dt_iceon_avg_prior
icemovAvg_prior
temp_init00
temp_heat01
temp_ice02
temp_diff03
temp_conv04
temp_mix05
buoyancy
diffusivity
densityGradient
tempChange
ice
snow
snowice
Volume_m2
Osgood
MaxDepth_m
MeanDepth_m
obs_temp


In [21]:
final_df_null.to_csv("erken-all_data_lake_modeling_in_time.csv", index=False)